# Phase 1, 80/20 Split: Unconstrained & Regularised Models
---
All 39 shape parameters. Single train/test split.
Metrics: R², MSE, MAE, clinical tolerance ±0.5 and ±1.0 BCS.

In [8]:
import warnings

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin
from mord import LogisticAT, LogisticIT

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_excel('COMBINED_HORSES_DATA.xlsx')

beta_cols = [f'beta_{i}' for i in range(0, 39)]
X = df[beta_cols]
y = df['BCS']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

Train: 166  |  Test: 42


## Helper, OrdinalWrapper
---
mord requires integer labels and is not directly sklearn-compatible.
This wrapper makes it usable inside Pipeline / GridSearchCV.

In [10]:
# sklearn-compatible wrapper for mord ordinal regression models
class OrdinalWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, model_type='AT', alpha=1.0):
        self.model_type = model_type
        self.alpha = alpha

    # fit the ordinal model after rounding y to integers
    def fit(self, X, y):
        y_int = np.round(y).astype(int)
        if self.model_type == 'AT':
            self.model_ = LogisticAT(alpha=self.alpha)
        elif self.model_type == 'IT':
            self.model_ = LogisticIT(alpha=self.alpha)
        else:
            raise ValueError("model_type must be 'AT' or 'IT'")
        self.model_.fit(X, y_int)
        return self

    # return predictions cast to float
    def predict(self, X):
        return self.model_.predict(X).astype(float)

## Helper, evaluation function
---

In [11]:
# fit model and print train/test metrics for both splits
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    tr_pred = model.predict(X_tr)
    te_pred = model.predict(X_te)

    tr_pred = np.array(tr_pred).flatten()
    te_pred = np.array(te_pred).flatten()

    te_errors = np.abs(np.array(y_te) - te_pred)

    print(f"{name}")
    print(f"  Train R²:  {r2_score(y_tr, tr_pred):.4f}   Test R²:  {r2_score(y_te, te_pred):.4f}")
    print(f"  Train MSE: {mean_squared_error(y_tr, tr_pred):.4f}   Test MSE: {mean_squared_error(y_te, te_pred):.4f}")
    print(f"  Train MAE: {mean_absolute_error(y_tr, tr_pred):.4f}   Test MAE: {mean_absolute_error(y_te, te_pred):.4f}")
    print(f"  Test ±0.5: {np.mean(te_errors <= 0.5):.1%}   Test ±1.0: {np.mean(te_errors <= 1.0):.1%}")
    print()

## Unconstrained Models
---
No regularisation

In [12]:
# unconstrained models
# each model is wrapped in a pipeline so the scaler is fit only on training data

# 1. ols
ols_pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
evaluate("OLS (Unconstrained)", ols_pipe, X_train, y_train, X_test, y_test)

# 2. mlp (unconstrained, alpha=0)
mlp_unc_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', MLPRegressor(hidden_layer_sizes=(4, 4), activation='tanh',
                           solver='adam', alpha=0, max_iter=2000, random_state=42))
])
evaluate("MLP (Unconstrained)", mlp_unc_pipe, X_train, y_train, X_test, y_test)

# 3. random forest (unconstrained, no depth limit)
rf_unc_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42))
])
evaluate("Random Forest (Unconstrained)", rf_unc_pipe, X_train, y_train, X_test, y_test)

# 4. ordinal regression (unconstrained, alpha=0)
ord_unc_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', OrdinalWrapper(model_type='IT', alpha=0.0))
])
evaluate("Ordinal Regression (Unconstrained)", ord_unc_pipe, X_train, y_train, X_test, y_test)

OLS (Unconstrained)
  Train R²:  0.6546   Test R²:  -0.1498
  Train MSE: 0.2705   Test MSE: 0.7568
  Train MAE: 0.4148   Test MAE: 0.6738
  Test ±0.5: 52.4%   Test ±1.0: 76.2%

MLP (Unconstrained)
  Train R²:  0.6751   Test R²:  -0.1296
  Train MSE: 0.2544   Test MSE: 0.7435
  Train MAE: 0.3850   Test MAE: 0.7205
  Test ±0.5: 40.5%   Test ±1.0: 76.2%

Random Forest (Unconstrained)
  Train R²:  0.9074   Test R²:  -0.1240
  Train MSE: 0.0725   Test MSE: 0.7398
  Train MAE: 0.2057   Test MAE: 0.6626
  Test ±0.5: 47.6%   Test ±1.0: 83.3%

Ordinal Regression (Unconstrained)
  Train R²:  0.6230   Test R²:  -0.3747
  Train MSE: 0.2952   Test MSE: 0.9048
  Train MAE: 0.2952   Test MAE: 0.6667
  Test ±0.5: 45.2%   Test ±1.0: 88.1%



## Regularised Models, GridSearchCV
---
Each model is wrapped in a Pipeline so the scaler is re-fit
per CV fold inside GridSearchCV.

In [13]:
cv_folds = 5

# run GridSearchCV and evaluate the best model on the train/test split
def tune_and_evaluate(name, pipe, param_grid):
    gs = GridSearchCV(pipe, param_grid, cv=cv_folds,
                      scoring='neg_mean_squared_error', n_jobs=-1)
    gs.fit(X_train, y_train)
    best = gs.best_estimator_
    print(f"Best Params: {gs.best_params_}")
    evaluate(name, best, X_train, y_train, X_test, y_test)
    return best

# Lasso
best_lasso = tune_and_evaluate(
    "Lasso",
    Pipeline([('scaler', StandardScaler()), ('model', Lasso(random_state=42))]),
    {'model__alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]}
)

# Ridge
best_ridge = tune_and_evaluate(
    "Ridge",
    Pipeline([('scaler', StandardScaler()), ('model', Ridge(random_state=42))]),
    {'model__alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0],
     'model__solver': ['auto', 'svd', 'cholesky', 'lsqr']}
)

# Pruned random forest
best_rf = tune_and_evaluate(
    "Random Forest (Pruned)",
    Pipeline([('scaler', StandardScaler()), ('model', RandomForestRegressor(random_state=42))]),
    {'model__n_estimators': [100, 200],
     'model__max_depth': [3, 5, 7, 10],
     'model__min_samples_split': [2, 5, 10],
     'model__min_samples_leaf': [1, 2, 4]}
)

# Regularised mlp
best_mlp = tune_and_evaluate(
    "MLP (Regularised)",
    Pipeline([('scaler', StandardScaler()),
              ('model', MLPRegressor(max_iter=2000, random_state=42))]),
    {'model__hidden_layer_sizes': [(4,), (4, 4), (8,)],
     'model__activation': ['relu', 'tanh'],
     'model__alpha': [0.001, 0.01, 0.1, 1.0, 5.0],
     'model__solver': ['adam', 'lbfgs']}
)

# Regularised ordinal regression
best_ordinal = tune_and_evaluate(
    "Ordinal Regression (Regularised)",
    Pipeline([('scaler', StandardScaler()),
              ('model', OrdinalWrapper(model_type='AT'))]),
    {'model__alpha': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0]}
)

Best Params: {'model__alpha': 0.01}
Lasso
  Train R²:  0.6358   Test R²:  0.0042
  Train MSE: 0.2852   Test MSE: 0.6554
  Train MAE: 0.4187   Test MAE: 0.6248
  Test ±0.5: 57.1%   Test ±1.0: 78.6%

Best Params: {'model__alpha': 10.0, 'model__solver': 'lsqr'}
Ridge
  Train R²:  0.6406   Test R²:  -0.0210
  Train MSE: 0.2814   Test MSE: 0.6720
  Train MAE: 0.4221   Test MAE: 0.6274
  Test ±0.5: 57.1%   Test ±1.0: 78.6%

Best Params: {'model__max_depth': 7, 'model__min_samples_leaf': 4, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Random Forest (Pruned)
  Train R²:  0.7944   Test R²:  -0.0954
  Train MSE: 0.1610   Test MSE: 0.7209
  Train MAE: 0.3018   Test MAE: 0.6555
  Test ±0.5: 45.2%   Test ±1.0: 85.7%

Best Params: {'model__activation': 'tanh', 'model__alpha': 5.0, 'model__hidden_layer_sizes': (4, 4), 'model__solver': 'lbfgs'}
MLP (Regularised)
  Train R²:  0.8539   Test R²:  -0.3108
  Train MSE: 0.1144   Test MSE: 0.8627
  Train MAE: 0.2750   Test MAE: 0.7516
  Test ±0

## VotingRegressor Ensemble
---
Ordinal Regression is excluded: it outputs integer classes which are
incompatible with continuous averaging in VotingRegressor.

All sub-models already contain an internal StandardScaler, so raw
(unscaled) X_train / X_test is passed.

In [14]:
ensemble = VotingRegressor(estimators=[
    ('lasso', best_lasso),
    ('ridge', best_ridge),
    ('rf', best_rf),
    ('mlp', best_mlp),
])

evaluate("Voting Regressor (Lasso + Ridge + RF + MLP)", ensemble, X_train, y_train, X_test, y_test)

Voting Regressor (Lasso + Ridge + RF + MLP)
  Train R²:  0.7624   Test R²:  -0.0476
  Train MSE: 0.1861   Test MSE: 0.6895
  Train MAE: 0.3383   Test MAE: 0.6513
  Test ±0.5: 52.4%   Test ±1.0: 83.3%

